In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cartopy.crs as ccrs
import cmocean
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import Rectangle
from datetime import datetime 
import dask

In [2]:
from OceanDataStore import OceanDataCatalog

In [3]:
catalog = OceanDataCatalog(catalog_name="noc-model-stac")

In [4]:
catalog.available_collections

['noc-rapid-evolution', 'noc-npd-jra55', 'noc-npd-era5']

In [5]:
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')


            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1y
              Title: eORCA1 ERA5v1 NPD T1y Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics annual mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2024-12-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1m
              Title: eORCA1 ERA5v1 NPD T1m Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics monthly mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2024-12-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d
              Title: eORCA025 ERA5v1 NPD T1y_3d Icechunk repository
              Description: Icechunk repository containing eORCA025 ERA5v1 

In [6]:
catalog.available_items

['noc-npd-era5/npd-eorca1-era5v1/gn/T1y',
 'noc-npd-era5/npd-eorca1-era5v1/gn/T1m',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T1m_3d',
 'noc-npd-era5/npd-eorca025-era5v1/gn/T5d_3d',
 'noc-npd-era5/npd-eorca12-era5v1/gn/T1y_3d',
 'noc-npd-era5/npd-eorca12-era5v1/gn/T1m_3d']

In [7]:
catalog.Items[3]

<Item id=noc-npd-era5/npd-eorca025-era5v1/gn/T1m_3d>

In [8]:
ds1 = catalog.open_dataset(id=catalog.Items[3].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12', 
                          bbox = (-85.0, 0.0, 0.0, 80.0))

In [9]:
## Preparing data array of trends form month number 

def get_trend_data (month_number):
    months = []
    for year in range (1976, 2025):
        months.append(f'{year}-{month_number:02d}-15')
    monthly_data = ds1['tos_con'].sel(time_counter=months, method='nearest')
    monthly_data['time_counter'] = monthly_data['time_counter'].dt.year
    monthly_data = monthly_data.compute()

    ny, nx = monthly_data.sizes['y'], monthly_data.sizes['x']
    trend_data = np.full((ny, nx), np.nan, dtype=np.float32)
    for y_idx in range (ny):
        for x_idx in range (nx):
            point = monthly_data.isel(y=y_idx, x=x_idx)
            try:
                z = np.polyfit(point['time_counter'], point.values, 1)
                trend_data[y_idx, x_idx] = z[0]
            except:
                trend_data[y_idx, x_idx] = np.nan

    trend_da = xr.DataArray(data = trend_data, dims=["y", "x"], 
            coords={ "y": monthly_data['y'],
            "x": monthly_data['x'],
            "nav_lat": (("y", "x"), monthly_data['nav_lat'].values),
            "nav_lon": (("y", "x"), monthly_data['nav_lon'].values)}, name="trend",
            attrs={"description": "Linear trend over time of SST"})
    
    return trend_da 
    

In [10]:
monthly_list = [ get_trend_data(m) for m in range(1,13) ]


NameError: name 'months' is not defined

In [11]:
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

for i, da in enumerate(monthly_list):
    arr = da.values
    print(i+1, months[i],
          "shape:", arr.shape,
          "min:", np.nanmin(arr),
          "median:", np.nanmedian(arr),
          "max:", np.nanmax(arr))
    # print a tiny sample
    print(" sample[0:3,0:3]:\n", np.round(arr[:3,:3], 4))
    print("-"*60)

1 January shape: (482, 341) min: -0.10307141 median: 0.013526035 max: 0.28559265
 sample[0:3,0:3]:
 [[   nan 0.0319 0.0332]
 [   nan 0.0298 0.0309]
 [   nan 0.0288 0.0299]]
------------------------------------------------------------
2 February shape: (482, 341) min: -0.150239 median: 0.002032472 max: 0.3201486
 sample[0:3,0:3]:
 [[   nan 0.0543 0.0553]
 [   nan 0.0516 0.0521]
 [   nan 0.0487 0.049 ]]
------------------------------------------------------------
3 March shape: (482, 341) min: -0.18639125 median: 0.00041490304 max: 0.26495132
 sample[0:3,0:3]:
 [[   nan 0.0533 0.055 ]
 [   nan 0.0476 0.0491]
 [   nan 0.0433 0.0446]]
------------------------------------------------------------
4 April shape: (482, 341) min: -0.16508119 median: 0.009069272 max: 0.27797166
 sample[0:3,0:3]:
 [[   nan 0.0117 0.0127]
 [   nan 0.0103 0.0116]
 [   nan 0.0107 0.0115]]
------------------------------------------------------------
5 May shape: (482, 341) min: -0.11357947 median: 0.033496536 max: 0.

In [14]:
monthly_list[6]

<xarray.DataArray 'trend' (y: 482, x: 341)> Size: 657kB
array([[        nan,  0.01141121,  0.01279766, ..., -0.06399174,
        -0.06500851, -0.06585299],
       [        nan,  0.0147902 ,  0.01601286, ..., -0.0609399 ,
        -0.06169927, -0.06216481],
       [        nan,  0.01612186,  0.01760524, ..., -0.0571837 ,
        -0.05753125, -0.05756548],
       ...,
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan]], shape=(482, 341), dtype=float32)
Coordinates:
  * y        (y) int64 4kB 0 1 2 3 4 5 6 7 8 ... 474 475 476 477 478 479 480 481
  * x        (x) int64 3kB 0 1 2 3 4 5 6 7 8 ... 333 334 335 336 337 338 339 340
    nav_lat  (y, x) float64 1MB 0.0 0.0 0.0 0.0 0.0 ... 79.51 79.41 79.3 79.2
    nav_lon  (y, x) float64 1MB -85.0 -84.75 -84.5 -84.25 ... 48.28 48.54 48.79
Attributes:
    description:  Linear trend over time of SST

In [ ]:
## Plotting Animation of monthly trends

months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
jan_data = get_trend_data(1)

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 6))
im = ax.pcolormesh(jan_data['nav_lon'], jan_data['nav_lat'], jan_data, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin= -0.08, vmax=0.08)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im, ax=ax, label='Trend in SST, degrees C / year ')
plt.tight_layout()
title = ax.set_title(f'{months[0]} SST Trend: 1990 - 2024')

def update(frame):
    global im
    im.remove()
    monthly_data = get_trend_data(frame + 1)
    im = ax.pcolormesh(monthly_data['nav_lon'], monthly_data['nav_lat'], monthly_data, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin= -0.08, vmax=0.08)  
    title.set_text(f'{months[frame]} SST Trend: 1990 - 2024')
    print(months[frame])
    return [im, title]

plt.close(fig)
anim = FuncAnimation(fig, update, frames = 12, interval=1000, repeat=True)
anim.save('Trends_by_month.gif', writer='pillow', fps=1)
HTML(anim.to_jshtml())


January
January
February
March


In [16]:
monthly_list[0]

<xarray.DataArray 'trend' (y: 482, x: 341)> Size: 657kB
array([[       nan, 0.03185784, 0.03324274, ..., 0.01822737, 0.01686788,
        0.01559964],
       [       nan, 0.02980281, 0.03091074, ..., 0.01755708, 0.0167582 ,
        0.01585856],
       [       nan, 0.0287914 , 0.02988294, ..., 0.0179816 , 0.01734134,
        0.01692334],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan]], shape=(482, 341), dtype=float32)
Coordinates:
  * y        (y) int64 4kB 0 1 2 3 4 5 6 7 8 ... 474 475 476 477 478 479 480 481
  * x        (x) int64 3kB 0 1 2 3 4 5 6 7 8 ... 333 334 335 336 337 338 339 340
    nav_lat  (y, x) float64 1MB 0.0 0.0 0.0 0.0 0.0 ... 79.51 79.41 79.3 79.2
    nav_lon  (y, x) float64 1MB -85.0 -84.75 -84.5 -84.25 ... 48.28 48.54 48.79
Attributes:
    description:  Linear trend over time of SST